In [1]:
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ----- 1) Imports & Reproducibility -----
import os
import math
import random
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [29]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat/landsat_features_training_mvdb.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

# Fix Dataset
eps = 1e-10
landsat_train_features['Ndmi'] = (landsat_train_features['Nir08'] - landsat_train_features['Swir16']) / (landsat_train_features['Nir08'] + landsat_train_features['Swir16'] + eps)
landsat_train_features['Mndwi'] = (landsat_train_features['Green'] - landsat_train_features['Swir16']) / (landsat_train_features['Green'] + landsat_train_features['Swir16'] + eps)

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)

#wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date',  'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#, 'Atmos opacity, 'Emsd', 'Lwir''
feature_cols = wq_data.columns.tolist()

#print(wq_data.columns)
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')

wq_data = wq_data.dropna(subset=feature_cols)

wq_data = wq_data.drop(columns='Sample Date')

In [31]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat/landsat_features_validation_mvdb.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

# Fix Dataset
eps = 1e-10
landsat_train_features_v['Ndmi'] = (landsat_train_features_v['Nir08'] - landsat_train_features_v['Swir16']) / (landsat_train_features_v['Nir08'] + landsat_train_features_v['Swir16'] + eps)
landsat_train_features_v['Mndwi'] = (landsat_train_features_v['Green'] - landsat_train_features_v['Swir16']) / (landsat_train_features_v['Green'] + landsat_train_features_v['Swir16'] + eps)

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

#wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)

wq_data_final = wq_data_v[['Latitude', 'Longitude', 'Sample Date', 'Month_cosine']]


wq_data_v = wq_data_v.dropna(subset=feature_cols)

wq_data_v = wq_data_v.drop(columns='Sample Date')

wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

print(wq_data_final.columns)

In [32]:
# For reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [33]:
#number of cv groups
cv_groups = 8

#split over longitude
wq_data['cv_group'] = pd.qcut(wq_data['Latitude'], q=cv_groups, labels=False)

## Specify the number of folds
#lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [ ]:
ava_cols = wq_data.columns.tolist()

#print(wq_data.columns)
ava_cols.remove('Latitude')
ava_cols.remove('Longitude')
ava_cols.remove('Month_cosine')
ava_cols.remove('Total Alkalinity')
ava_cols.remove('Electrical Conductance')
ava_cols.remove('Dissolved Reactive Phosphorus')

#'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil'

print(ava_cols)

In [35]:


# 2) Helper to return a DataFrame with scaled features but keys/targets intact
def apply_scaler(df, key_cols, feature_cols, target_cols, scaler):
    df_out = df.copy()
    X_scaled = scaler.transform(df_out[feature_cols])
    df_out.loc[:, feature_cols] = X_scaled  # replace only features
    return df_out


# ----- 7) PyTorch Dataset -----
class TabularDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, key_cols, feature_cols, target_cols, task_type="regression"):
        self.keys = frame[key_cols].reset_index(drop=True)
        X = frame[feature_cols].to_numpy(dtype=np.float32)
        self.X = torch.from_numpy(X)
        y = frame[target_cols].to_numpy(dtype=np.float32)
        self.y = torch.from_numpy(y)
        self.task_type = task_type

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Return keys (as dict-like), features, and targets
        return self.keys.iloc[idx].to_dict(), self.X[idx], self.y[idx]
    
# ----- 9) Model: a simple MLP -----
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dims=(128, 64), dropout=0.1, task_type="regression"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
        self.task_type = task_type

    def forward(self, x):
        logits_or_outputs = self.net(x)
        # For regression, return raw outputs.
        # For multilabel classification: during training we keep raw logits (BCEWithLogitsLoss expects logits).
        return logits_or_outputs

In [36]:
# ----- 11) Metrics -----
@torch.no_grad()
def evaluate(model, loader, task_type="regression", threshold=0.5):

    preds_list = []
    targets_list = []


    model.eval()
    total_loss = 0.0
    n_examples = 0

    # For regression: track MAE and RMSE
    mae_sum = torch.zeros(len(target_cols), device=DEVICE)
    mse_sum = torch.zeros(len(target_cols), device=DEVICE)

    # For multilabel: track accuracy (micro)
    correct = 0
    total   = 0

    for _, X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        out = model(X)

        preds_list.append(out.detach().cpu())
        targets_list.append(y.detach().cpu())

        loss = criterion(out, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print("Bad loss detected!")
            print("out min/max:", out.min().item(), out.max().item())
            print("y min/max:", y.min().item(), y.max().item())
            print("loss:", loss)
            return {"loss": float("nan")}

        bs = y.size(0)
        total_loss += loss.item() * bs
        n_examples += bs

        if task_type == "regression":
            err = torch.abs(out - y)
            mae_sum += err.sum(dim=0)
            mse_sum += ((out - y) ** 2).sum(dim=0)
        else:
            # multilabel: apply sigmoid then threshold
            probs = torch.sigmoid(out)
            preds = (probs >= threshold).float()
            correct += (preds == y).sum().item()
            total   += y.numel()

    avg_loss = total_loss / max(n_examples, 1)

    if task_type == "regression":

        preds = torch.cat(preds_list, dim=0)
        targets = torch.cat(targets_list, dim=0)

        mae = (mae_sum / n_examples).detach().cpu().numpy()
        rmse = torch.sqrt(mse_sum / n_examples).detach().cpu().numpy()

        # Compute R2 per target
        ss_res = torch.sum((preds - targets) ** 2, dim=0)
        ss_tot = torch.sum((targets - targets.mean(dim=0)) ** 2, dim=0)

        r2 = 1 - ss_res / torch.clamp(ss_tot, min=1e-12)
        r2 = r2.numpy()

        metrics = {
            "loss": avg_loss,
            "MAE_per_target": {t: float(mae[i]) for i, t in enumerate(target_cols)},
            "RMSE_per_target": {t: float(rmse[i]) for i, t in enumerate(target_cols)},
            "R2_per_target": {t: float(r2[i]) for i, t in enumerate(target_cols)},
        }
    else:
        accuracy = correct / max(total, 1)
        metrics = {"loss": avg_loss, "micro_accuracy": accuracy}

    return metrics

In [ ]:
def metrics_to_dataframe(metrics):
    df = pd.DataFrame({
        "MAE": metrics["MAE_per_target"],
        "RMSE": metrics["RMSE_per_target"],
        "R2": metrics["R2_per_target"]
    })

    # Add overall loss as a separate column (same value for all rows)
    df["Loss"] = metrics["loss"]

    df = df.T

    return df


In [45]:
all_metrics = []

for i in ava_cols:
   
    if i in ['Swir22', 'Ndmi', 'Mndwi', 'Pet', 'Aet', 'Soil', 'Def', 'cv_group']:
      continue
    
    print(i)
    
    wq_data_mini = wq_data[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
      'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Swir22', 'Ndmi', 'Mndwi', 'Pet', 'Aet', 'Soil', 'Def', 'cv_group', i]]
    wq_data_v_mini = wq_data_v[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
      'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Swir22', 'Ndmi', 'Mndwi', 'Pet', 'Aet', 'Soil', 'Def',i]]
    
    #test train based on location
    wq_data_test = wq_data_mini[wq_data_mini['cv_group'] == 7]
    wq_data_train = wq_data_mini[wq_data_mini['cv_group'] != 7]
    
    wq_data_test = wq_data_test.drop(columns='cv_group')
    wq_data_train = wq_data_train.drop(columns='cv_group')
    
    wq_data_train = wq_data_train[sorted(wq_data_train.columns)]
    wq_data_test = wq_data_test[sorted(wq_data_test.columns)]
    
    # ----- 2) Configurable column names & task type -----
    # Adjust these lists for your real dataset
    key_cols     = ["Latitude", "Longitude", "Month_cosine"]  # 3 key columns (IDs, timestamps, etc.)
    feature_cols = wq_data_mini.columns.tolist()
    feature_cols.remove('Latitude')
    feature_cols.remove('Longitude')
    feature_cols.remove('Month_cosine')
    feature_cols.remove('Total Alkalinity')
    feature_cols.remove('Electrical Conductance')
    feature_cols.remove('Dissolved Reactive Phosphorus')
    feature_cols.remove('cv_group')
    #feature_cols = [f"f{i}" for i in range(1, 21)]  # 20 feature columns
    target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
    #target_cols  = ["y1", "y2", "y3"]  # 3 target columns
    
    # Choose the task: "regression" (default) or "multilabel"
    #  - regression:  continuous y1,y2,y3  -> MSELoss
    #  - multilabel:  binary y1,y2,y3 in {0,1} -> BCEWithLogitsLoss
    task_type = "regression"
    
    # # ----- 6) Compute normalization stats on TRAIN ONLY -----
    # # Standardize features to zero-mean, unit-std
    # feat_means = wq_data_train[feature_cols].mean()
    # feat_stds  = wq_data_train[feature_cols].std().replace(0, 1.0)  # avoid division by zero
    
    # 1) Fit on TRAIN FEATURES ONLY
    scaler = StandardScaler()
    scaler.fit(wq_data_train[feature_cols])


    df_train_n = apply_scaler(wq_data_train, key_cols, feature_cols, target_cols, scaler)
    df_val_n   = apply_scaler(wq_data_test,  key_cols, feature_cols, target_cols, scaler)
    df_test_n  = apply_scaler(wq_data_v_mini, key_cols, feature_cols, target_cols, scaler)
    
    # Now this works because df_*_n are DataFrames
    train_ds = TabularDataset(df_train_n, key_cols, feature_cols, target_cols, task_type=task_type)
    val_ds   = TabularDataset(df_val_n,   key_cols, feature_cols, target_cols, task_type=task_type)
    test_ds  = TabularDataset(df_test_n,  key_cols, feature_cols, target_cols, task_type=task_type)

    # ----- 8) DataLoaders -----
    BATCH_SIZE = 128
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    
    model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols), hidden_dims=(128, 64), dropout=0.15, task_type=task_type).to(DEVICE)
    
    # ----- 10) Loss and Optimizer -----
    if task_type == "regression":
      criterion = nn.MSELoss()
    else:  # "multilabel"
      criterion = nn.BCEWithLogitsLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    
    # ----- 12) Training Loop -----
    EPOCHS = 15
    best_val = math.inf
    patience = 10
    wait = 0
    best_state = None
    
    
    
    for epoch in range(1, EPOCHS + 1):
      model.train()
      running_loss = 0.0
      n_seen = 0
    
      for _, X, y in train_loader:
         X = X.to(DEVICE)
         y = y.to(DEVICE)
    
         optimizer.zero_grad()
         out = model(X)
         
         loss = criterion(out, y)
         loss.backward()
         optimizer.step()
    
         running_loss += loss.item() * X.size(0)
         n_seen += X.size(0)
    
      train_loss = running_loss / max(n_seen, 1)
      val_metrics = evaluate(model, val_loader, task_type=task_type)
      val_loss = val_metrics["loss"]
    
      #print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Metrics: {val_metrics}")
    
      # Simple early stopping on validation loss
      if val_loss < best_val - 1e-6:
         best_val = val_loss
         best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
         wait = 0
      else:
         wait += 1
         if wait >= patience:
               #print("Early stopping.")
               break
    
    # Load best model (if available)
    if best_state is not None:
      model.load_state_dict(best_state)
    
    test_metrics = evaluate(model, val_loader, task_type=task_type)
    print("TEST:", test_metrics)
    
    df_metrics = metrics_to_dataframe(test_metrics)
    
    df_metrics = df_metrics.rename(columns={'Total Alkalinity': i})
    
    all_metrics.append(df_metrics)

In [46]:
all_metrics_copy = all_metrics

In [47]:
print(all_metrics_copy)

In [19]:
# ----- 14) Inference: get predictions + keys side-by-side -----
@torch.no_grad()
def predict_with_keys(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    all_rows = []
    for keys, X, _ in loader:
        X = X.to(DEVICE)
        logits_or_outputs = model(X).cpu().numpy()

        if task_type == "regression":
            preds = logits_or_outputs  # raw outputs
        else:
            probs = 1 / (1 + np.exp(-logits_or_outputs))  # sigmoid
            preds = (probs >= threshold).astype(np.float32)

        # For each batch row, combine keys + predictions into a dict
        for i in range(len(keys["Latitude"])):
            row = {k: keys[k][i] for k in keys.keys()}
            for j, t in enumerate(target_cols):
                row[f"pred_{t}"] = float(preds[i, j])
            all_rows.append(row)
    return pd.DataFrame(all_rows)

pred_df = predict_with_keys(model, test_loader, task_type=task_type)
pred_df.head(5)

In [20]:
pred_df['Latitude'] = pred_df['Latitude'].apply(lambda x: x.item())
pred_df['Longitude'] = pred_df['Longitude'].apply(lambda x: x.item())
pred_df['Month_cosine'] = pred_df['Month_cosine'].apply(lambda x: x.item())

In [21]:
# Reverse cosine encoding
def reverse_month_cosine(cos_val):
    # Ensure value is in valid range for arccos
    cos_val = np.clip(cos_val, -1, 1)
    
    # Get angle in radians
    angle = np.arccos(cos_val)
    
    # Convert back to fractional month
    fractional_month = angle / (np.pi / 6)  # inverse of * np.pi/6
    
    # Approximate month and day
    month = int(fractional_month)  # integer month
    day = int(round((fractional_month - month) * 31))  # approximate day
    
    # Adjust for month starting at 1
    month = max(1, min(12, month))
    day = max(1, min(31, day))
    
    return month, day

# # Apply to DataFrame
# pred_df[['Approx_Month', 'Approx_Day']] = pred_df['Month_cosine'].apply(
#     lambda x: pd.Series(reverse_month_cosine(x))
# )

In [22]:
display(pred_df)

In [ ]:
# # ----- 15) (Optional) Save artifacts -----
# os.makedirs("artifacts", exist_ok=True)
# torch.save({
#     "model_state_dict": model.state_dict(),
#     "feature_cols": feature_cols,
#     "target_cols": target_cols,
#     "feat_means": feat_means.to_dict(),
#     "feat_stds": feat_stds.to_dict(),
#     "task_type": task_type
# }, "artifacts/model.pt")

# pred_df.to_csv("artifacts/test_predictions_with_keys.csv", index=False)
# #print("Saved: artifacts/model.pt and artifacts/test_predictions_with_keys.csv")

In [23]:
condition = (pred_df['pred_Total Alkalinity'] > 20) & (pred_df['pred_Total Alkalinity'] < 200)

count1 = condition.sum()

print("Count using sum():", count1)


condition = (pred_df['pred_Electrical Conductance'] < 800)

count1 = condition.sum()

print("Count using sum():", count1)

condition = (pred_df['pred_Dissolved Reactive Phosphorus'] < 100)

count1 = condition.sum()

print("Count using sum():", count1)

In [135]:
pred_df.to_csv('data/output.csv', index=False)

In [48]:
import ast
import pandas as pd

raw_text = """
Emis
TEST: {'loss': 31293.512396869348, 'MAE_per_target': {'Total Alkalinity': 55.214595794677734, 'Electrical Conductance': 230.62205505371094, 'Dissolved Reactive Phosphorus': 37.47528839111328}, 'RMSE_per_target': {'Total Alkalinity': 68.47237396240234, 'Electrical Conductance': 294.2490539550781, 'Dissolved Reactive Phosphorus': 51.083919525146484}, 'R2_per_target': {'Total Alkalinity': -0.0240018367767334, 'Electrical Conductance': 0.09892773628234863, 'Dissolved Reactive Phosphorus': 0.11092549562454224}}
Atran
TEST: {'loss': 31694.250558035714, 'MAE_per_target': {'Total Alkalinity': 55.5501708984375, 'Electrical Conductance': 230.73977661132812, 'Dissolved Reactive Phosphorus': 36.82624435424805}, 'RMSE_per_target': {'Total Alkalinity': 67.86968231201172, 'Electrical Conductance': 296.39483642578125, 'Dissolved Reactive Phosphorus': 51.24999237060547}, 'R2_per_target': {'Total Alkalinity': -0.0060547590255737305, 'Electrical Conductance': 0.08573782444000244, 'Dissolved Reactive Phosphorus': 0.10513532161712646}}
Swir16
TEST: {'loss': 31253.5984403255, 'MAE_per_target': {'Total Alkalinity': 54.300724029541016, 'Electrical Conductance': 229.0863800048828, 'Dissolved Reactive Phosphorus': 37.918514251708984}, 'RMSE_per_target': {'Total Alkalinity': 67.52259826660156, 'Electrical Conductance': 294.3994445800781, 'Dissolved Reactive Phosphorus': 50.30363082885742}, 'R2_per_target': {'Total Alkalinity': 0.004208803176879883, 'Electrical Conductance': 0.09800630807876587, 'Dissolved Reactive Phosphorus': 0.13787859678268433}}
Atmos_Opacity
TEST: {'loss': 31078.72785516501, 'MAE_per_target': {'Total Alkalinity': 54.87120819091797, 'Electrical Conductance': 228.126220703125, 'Dissolved Reactive Phosphorus': 37.424232482910156}, 'RMSE_per_target': {'Total Alkalinity': 67.75325012207031, 'Electrical Conductance': 293.3460693359375, 'Dissolved Reactive Phosphorus': 50.92914962768555}, 'R2_per_target': {'Total Alkalinity': -0.002605915069580078, 'Electrical Conductance': 0.10444962978363037, 'Dissolved Reactive Phosphorus': 0.11630469560623169}}
Green
TEST: {'loss': 31349.119324282325, 'MAE_per_target': {'Total Alkalinity': 54.05317687988281, 'Electrical Conductance': 230.34906005859375, 'Dissolved Reactive Phosphorus': 37.5067253112793}, 'RMSE_per_target': {'Total Alkalinity': 67.7215347290039, 'Electrical Conductance': 294.8103332519531, 'Dissolved Reactive Phosphorus': 50.477848052978516}, 'R2_per_target': {'Total Alkalinity': -0.001667618751525879, 'Electrical Conductance': 0.09548681974411011, 'Dissolved Reactive Phosphorus': 0.13189679384231567}}
Drad
TEST: {'loss': 31348.526139381782, 'MAE_per_target': {'Total Alkalinity': 55.06270217895508, 'Electrical Conductance': 228.90786743164062, 'Dissolved Reactive Phosphorus': 38.2099723815918}, 'RMSE_per_target': {'Total Alkalinity': 67.77275085449219, 'Electrical Conductance': 294.7822265625, 'Dissolved Reactive Phosphorus': 50.55547332763672}, 'R2_per_target': {'Total Alkalinity': -0.0031833648681640625, 'Electrical Conductance': 0.09565931558609009, 'Dissolved Reactive Phosphorus': 0.1292247176170349}}
Red
TEST: {'loss': 31479.30014198124, 'MAE_per_target': {'Total Alkalinity': 54.704227447509766, 'Electrical Conductance': 230.7255401611328, 'Dissolved Reactive Phosphorus': 38.1240234375}, 'RMSE_per_target': {'Total Alkalinity': 67.9610366821289, 'Electrical Conductance': 295.44427490234375, 'Dissolved Reactive Phosphorus': 50.31782913208008}, 'R2_per_target': {'Total Alkalinity': -0.008764982223510742, 'Electrical Conductance': 0.09159272909164429, 'Dissolved Reactive Phosphorus': 0.13739192485809326}}
Emsd
TEST: {'loss': 31838.207059504974, 'MAE_per_target': {'Total Alkalinity': 54.58292007446289, 'Electrical Conductance': 231.2779998779297, 'Dissolved Reactive Phosphorus': 37.65653991699219}, 'RMSE_per_target': {'Total Alkalinity': 68.11273193359375, 'Electrical Conductance': 297.1127014160156, 'Dissolved Reactive Phosphorus': 50.983585357666016}, 'R2_per_target': {'Total Alkalinity': -0.013273119926452637, 'Electrical Conductance': 0.08130377531051636, 'Dissolved Reactive Phosphorus': 0.1144145131111145}}
Cdist
TEST: {'loss': 30929.02142080131, 'MAE_per_target': {'Total Alkalinity': 54.278987884521484, 'Electrical Conductance': 229.4930877685547, 'Dissolved Reactive Phosphorus': 36.54465866088867}, 'RMSE_per_target': {'Total Alkalinity': 67.31689453125, 'Electrical Conductance': 292.8320617675781, 'Dissolved Reactive Phosphorus': 50.04866409301758}, 'R2_per_target': {'Total Alkalinity': 0.010266900062561035, 'Electrical Conductance': 0.1075851321220398, 'Dissolved Reactive Phosphorus': 0.14659583568572998}}
Nir08
TEST: {'loss': 31459.450327757684, 'MAE_per_target': {'Total Alkalinity': 55.035362243652344, 'Electrical Conductance': 230.27842712402344, 'Dissolved Reactive Phosphorus': 37.66508102416992}, 'RMSE_per_target': {'Total Alkalinity': 67.8979263305664, 'Electrical Conductance': 295.2528076171875, 'Dissolved Reactive Phosphorus': 50.93128967285156}, 'R2_per_target': {'Total Alkalinity': -0.006892204284667969, 'Electrical Conductance': 0.0927695631980896, 'Dissolved Reactive Phosphorus': 0.11623024940490723}}
Blue
TEST: {'loss': 31521.02115591094, 'MAE_per_target': {'Total Alkalinity': 54.64247512817383, 'Electrical Conductance': 230.741455078125, 'Dissolved Reactive Phosphorus': 37.299285888671875}, 'RMSE_per_target': {'Total Alkalinity': 67.8310317993164, 'Electrical Conductance': 295.5921630859375, 'Dissolved Reactive Phosphorus': 50.86539840698242}, 'R2_per_target': {'Total Alkalinity': -0.0049092769622802734, 'Electrical Conductance': 0.09068310260772705, 'Dissolved Reactive Phosphorus': 0.1185154914855957}}
Trad
TEST: {'loss': 31339.449861550635, 'MAE_per_target': {'Total Alkalinity': 55.0175895690918, 'Electrical Conductance': 229.53421020507812, 'Dissolved Reactive Phosphorus': 37.1357307434082}, 'RMSE_per_target': {'Total Alkalinity': 67.6227035522461, 'Electrical Conductance': 294.7232666015625, 'Dissolved Reactive Phosphorus': 50.83010482788086}, 'R2_per_target': {'Total Alkalinity': 0.001253962516784668, 'Electrical Conductance': 0.09602093696594238, 'Dissolved Reactive Phosphorus': 0.11973828077316284}}
Lwir
TEST: {'loss': 31922.79809702758, 'MAE_per_target': {'Total Alkalinity': 55.80376434326172, 'Electrical Conductance': 232.14053344726562, 'Dissolved Reactive Phosphorus': 37.605892181396484}, 'RMSE_per_target': {'Total Alkalinity': 68.40162658691406, 'Electrical Conductance': 297.3980407714844, 'Dissolved Reactive Phosphorus': 51.42000961303711}, 'R2_per_target': {'Total Alkalinity': -0.02188694477081299, 'Electrical Conductance': 0.07953834533691406, 'Dissolved Reactive Phosphorus': 0.09918814897537231}}
Urad
TEST: {'loss': 31719.830939901673, 'MAE_per_target': {'Total Alkalinity': 55.712032318115234, 'Electrical Conductance': 230.98703002929688, 'Dissolved Reactive Phosphorus': 37.22027587890625}, 'RMSE_per_target': {'Total Alkalinity': 68.53671264648438, 'Electrical Conductance': 296.5008544921875, 'Dissolved Reactive Phosphorus': 50.49212646484375}, 'R2_per_target': {'Total Alkalinity': -0.025927305221557617, 'Electrical Conductance': 0.08508360385894775, 'Dissolved Reactive Phosphorus': 0.13140565156936646}}
Nvdi
TEST: {'loss': 31358.609777633363, 'MAE_per_target': {'Total Alkalinity': 54.7814826965332, 'Electrical Conductance': 230.28152465820312, 'Dissolved Reactive Phosphorus': 37.13529586791992}, 'RMSE_per_target': {'Total Alkalinity': 67.86030578613281, 'Electrical Conductance': 294.82672119140625, 'Dissolved Reactive Phosphorus': 50.47787857055664}, 'R2_per_target': {'Total Alkalinity': -0.005776762962341309, 'Electrical Conductance': 0.0953863263130188, 'Dissolved Reactive Phosphorus': 0.13189560174942017}}
Savi
TEST: {'loss': 31574.46604458635, 'MAE_per_target': {'Total Alkalinity': 55.170654296875, 'Electrical Conductance': 231.13662719726562, 'Dissolved Reactive Phosphorus': 37.13835906982422}, 'RMSE_per_target': {'Total Alkalinity': 67.77193450927734, 'Electrical Conductance': 295.8416748046875, 'Dissolved Reactive Phosphorus': 51.069278717041016}, 'R2_per_target': {'Total Alkalinity': -0.003159046173095703, 'Electrical Conductance': 0.08914732933044434, 'Dissolved Reactive Phosphorus': 0.1114349365234375}}
Bsi
TEST: {'loss': 31200.358580328888, 'MAE_per_target': {'Total Alkalinity': 54.33160400390625, 'Electrical Conductance': 229.08961486816406, 'Dissolved Reactive Phosphorus': 37.432533264160156}, 'RMSE_per_target': {'Total Alkalinity': 67.64053344726562, 'Electrical Conductance': 294.05804443359375, 'Dissolved Reactive Phosphorus': 50.55380630493164}, 'R2_per_target': {'Total Alkalinity': 0.0007271170616149902, 'Electrical Conductance': 0.10009700059890747, 'Dissolved Reactive Phosphorus': 0.12928205728530884}}
Ndbi
TEST: {'loss': 31155.67415305719, 'MAE_per_target': {'Total Alkalinity': 54.726295471191406, 'Electrical Conductance': 227.8494110107422, 'Dissolved Reactive Phosphorus': 37.31451416015625}, 'RMSE_per_target': {'Total Alkalinity': 67.73414611816406, 'Electrical Conductance': 293.83538818359375, 'Dissolved Reactive Phosphorus': 50.39722442626953}, 'R2_per_target': {'Total Alkalinity': -0.0020406246185302734, 'Electrical Conductance': 0.1014595627784729, 'Dissolved Reactive Phosphorus': 0.13466745615005493}}
Tcwi
TEST: {'loss': 31174.048796338157, 'MAE_per_target': {'Total Alkalinity': 54.108917236328125, 'Electrical Conductance': 228.68519592285156, 'Dissolved Reactive Phosphorus': 37.95710754394531}, 'RMSE_per_target': {'Total Alkalinity': 67.50299072265625, 'Electrical Conductance': 294.03338623046875, 'Dissolved Reactive Phosphorus': 50.09844207763672}, 'R2_per_target': {'Total Alkalinity': 0.0047869086265563965, 'Electrical Conductance': 0.10024803876876831, 'Dissolved Reactive Phosphorus': 0.14489734172821045}}
Evi
TEST: {'loss': 31312.496454000906, 'MAE_per_target': {'Total Alkalinity': 54.778873443603516, 'Electrical Conductance': 229.46694946289062, 'Dissolved Reactive Phosphorus': 38.11427688598633}, 'RMSE_per_target': {'Total Alkalinity': 67.8282241821289, 'Electrical Conductance': 294.58465576171875, 'Dissolved Reactive Phosphorus': 50.56373596191406}, 'R2_per_target': {'Total Alkalinity': -0.00482630729675293, 'Electrical Conductance': 0.09687113761901855, 'Dissolved Reactive Phosphorus': 0.12894004583358765}}
Osavi
TEST: {'loss': 31640.543943546563, 'MAE_per_target': {'Total Alkalinity': 54.95713806152344, 'Electrical Conductance': 232.44293212890625, 'Dissolved Reactive Phosphorus': 37.35548782348633}, 'RMSE_per_target': {'Total Alkalinity': 68.1397933959961, 'Electrical Conductance': 296.18951416015625, 'Dissolved Reactive Phosphorus': 50.501258850097656}, 'R2_per_target': {'Total Alkalinity': -0.014078497886657715, 'Electrical Conductance': 0.08700406551361084, 'Dissolved Reactive Phosphorus': 0.13109123706817627}}
Gndvi
TEST: {'loss': 31531.539931340416, 'MAE_per_target': {'Total Alkalinity': 54.6563835144043, 'Electrical Conductance': 231.04678344726562, 'Dissolved Reactive Phosphorus': 38.26258850097656}, 'RMSE_per_target': {'Total Alkalinity': 67.98261260986328, 'Electrical Conductance': 295.65386962890625, 'Dissolved Reactive Phosphorus': 50.61406707763672}, 'R2_per_target': {'Total Alkalinity': -0.009405374526977539, 'Electrical Conductance': 0.09030336141586304, 'Dissolved Reactive Phosphorus': 0.12720483541488647}}
Gcvi
TEST: {'loss': 31752.99196499209, 'MAE_per_target': {'Total Alkalinity': 55.34592819213867, 'Electrical Conductance': 232.7830352783203, 'Dissolved Reactive Phosphorus': 36.586143493652344}, 'RMSE_per_target': {'Total Alkalinity': 68.06975555419922, 'Electrical Conductance': 296.601806640625, 'Dissolved Reactive Phosphorus': 51.505916595458984}, 'R2_per_target': {'Total Alkalinity': -0.011995077133178711, 'Electrical Conductance': 0.08446061611175537, 'Dissolved Reactive Phosphorus': 0.0961759090423584}}
Msi
TEST: {'loss': 31042.256858894667, 'MAE_per_target': {'Total Alkalinity': 54.54921340942383, 'Electrical Conductance': 227.53817749023438, 'Dissolved Reactive Phosphorus': 37.27915954589844}, 'RMSE_per_target': {'Total Alkalinity': 67.5090103149414, 'Electrical Conductance': 293.2108459472656, 'Dissolved Reactive Phosphorus': 50.957984924316406}, 'R2_per_target': {'Total Alkalinity': 0.004609465599060059, 'Electrical Conductance': 0.1052752137184143, 'Dissolved Reactive Phosphorus': 0.11530369520187378}}
Bui
TEST: {'loss': 31367.866671846747, 'MAE_per_target': {'Total Alkalinity': 54.7236328125, 'Electrical Conductance': 229.82469177246094, 'Dissolved Reactive Phosphorus': 36.90526580810547}, 'RMSE_per_target': {'Total Alkalinity': 67.36359405517578, 'Electrical Conductance': 294.8739929199219, 'Dissolved Reactive Phosphorus': 51.1378288269043}, 'R2_per_target': {'Total Alkalinity': 0.008893132209777832, 'Electrical Conductance': 0.09509623050689697, 'Dissolved Reactive Phosphorus': 0.10904794931411743}}
Tc Brightness
TEST: {'loss': 31700.331466150543, 'MAE_per_target': {'Total Alkalinity': 56.015506744384766, 'Electrical Conductance': 231.1729736328125, 'Dissolved Reactive Phosphorus': 37.6855583190918}, 'RMSE_per_target': {'Total Alkalinity': 68.5108413696289, 'Electrical Conductance': 296.3706359863281, 'Dissolved Reactive Phosphorus': 50.7119026184082}, 'R2_per_target': {'Total Alkalinity': -0.025152921676635742, 'Electrical Conductance': 0.08588701486587524, 'Dissolved Reactive Phosphorus': 0.12382769584655762}}
Tc Greenness
TEST: {'loss': 31808.36252896135, 'MAE_per_target': {'Total Alkalinity': 54.575077056884766, 'Electrical Conductance': 232.3483428955078, 'Dissolved Reactive Phosphorus': 36.54020690917969}, 'RMSE_per_target': {'Total Alkalinity': 68.07742309570312, 'Electrical Conductance': 296.9427490234375, 'Dissolved Reactive Phosphorus': 51.142547607421875}, 'R2_per_target': {'Total Alkalinity': -0.012223124504089355, 'Electrical Conductance': 0.08235454559326172, 'Dissolved Reactive Phosphorus': 0.10888338088989258}}
Nbr
TEST: {'loss': 31413.603413200723, 'MAE_per_target': {'Total Alkalinity': 55.26952362060547, 'Electrical Conductance': 230.57321166992188, 'Dissolved Reactive Phosphorus': 37.58891677856445}, 'RMSE_per_target': {'Total Alkalinity': 67.77386474609375, 'Electrical Conductance': 295.0425720214844, 'Dissolved Reactive Phosphorus': 50.964599609375}, 'R2_per_target': {'Total Alkalinity': -0.0032161474227905273, 'Electrical Conductance': 0.09406113624572754, 'Dissolved Reactive Phosphorus': 0.11507397890090942}}
Ndsi
TEST: {'loss': 31458.103526220613, 'MAE_per_target': {'Total Alkalinity': 55.22055435180664, 'Electrical Conductance': 230.54632568359375, 'Dissolved Reactive Phosphorus': 37.82529067993164}, 'RMSE_per_target': {'Total Alkalinity': 67.97077941894531, 'Electrical Conductance': 295.3533935546875, 'Dissolved Reactive Phosphorus': 50.206138610839844}, 'R2_per_target': {'Total Alkalinity': -0.009054303169250488, 'Electrical Conductance': 0.09215134382247925, 'Dissolved Reactive Phosphorus': 0.14121723175048828}}
Green/Red Ratio
TEST: {'loss': 31577.523147886528, 'MAE_per_target': {'Total Alkalinity': 54.581748962402344, 'Electrical Conductance': 229.88400268554688, 'Dissolved Reactive Phosphorus': 38.7362174987793}, 'RMSE_per_target': {'Total Alkalinity': 68.05032348632812, 'Electrical Conductance': 295.9523010253906, 'Dissolved Reactive Phosphorus': 50.139320373535156}, 'R2_per_target': {'Total Alkalinity': -0.011417388916015625, 'Electrical Conductance': 0.08846580982208252, 'Dissolved Reactive Phosphorus': 0.14350157976150513}}
Ndgi
TEST: {'loss': 31659.26527534471, 'MAE_per_target': {'Total Alkalinity': 54.66291427612305, 'Electrical Conductance': 230.71038818359375, 'Dissolved Reactive Phosphorus': 36.69974899291992}, 'RMSE_per_target': {'Total Alkalinity': 67.52218627929688, 'Electrical Conductance': 296.2735290527344, 'Dissolved Reactive Phosphorus': 51.38610076904297}, 'R2_per_target': {'Total Alkalinity': 0.004220783710479736, 'Electrical Conductance': 0.0864858627319336, 'Dissolved Reactive Phosphorus': 0.10037577152252197}}
Ui (Urban Index)
TEST: {'loss': 31208.97924319055, 'MAE_per_target': {'Total Alkalinity': 54.15949249267578, 'Electrical Conductance': 228.9634246826172, 'Dissolved Reactive Phosphorus': 38.333961486816406}, 'RMSE_per_target': {'Total Alkalinity': 67.50858306884766, 'Electrical Conductance': 294.2254638671875, 'Dissolved Reactive Phosphorus': 50.00908660888672}, 'R2_per_target': {'Total Alkalinity': 0.004622220993041992, 'Electrical Conductance': 0.09907227754592896, 'Dissolved Reactive Phosphorus': 0.14794498682022095}}
Nbr2
TEST: {'loss': 31098.333564082277, 'MAE_per_target': {'Total Alkalinity': 53.622161865234375, 'Electrical Conductance': 228.7692413330078, 'Dissolved Reactive Phosphorus': 37.50785446166992}, 'RMSE_per_target': {'Total Alkalinity': 67.12211608886719, 'Electrical Conductance': 293.6581115722656, 'Dissolved Reactive Phosphorus': 50.5424690246582}, 'R2_per_target': {'Total Alkalinity': 0.015986084938049316, 'Electrical Conductance': 0.1025434136390686, 'Dissolved Reactive Phosphorus': 0.1296725869178772}}
Red/Nir Ratio
TEST: {'loss': 31643.925428062837, 'MAE_per_target': {'Total Alkalinity': 55.36503982543945, 'Electrical Conductance': 231.5521697998047, 'Dissolved Reactive Phosphorus': 37.69910430908203}, 'RMSE_per_target': {'Total Alkalinity': 68.50187683105469, 'Electrical Conductance': 296.1014404296875, 'Dissolved Reactive Phosphorus': 50.62826156616211}, 'R2_per_target': {'Total Alkalinity': -0.024884581565856934, 'Electrical Conductance': 0.08754700422286987, 'Dissolved Reactive Phosphorus': 0.1267155408859253}}
Green/Nir Ratio
TEST: {'loss': 31323.590797355333, 'MAE_per_target': {'Total Alkalinity': 54.11235809326172, 'Electrical Conductance': 229.55697631835938, 'Dissolved Reactive Phosphorus': 38.58286666870117}, 'RMSE_per_target': {'Total Alkalinity': 67.47298431396484, 'Electrical Conductance': 294.8265075683594, 'Dissolved Reactive Phosphorus': 49.955101013183594}, 'R2_per_target': {'Total Alkalinity': 0.005671501159667969, 'Electrical Conductance': 0.09538775682449341, 'Dissolved Reactive Phosphorus': 0.1497836709022522}}
Aet_Roll3
TEST: {'loss': 30389.967471462478, 'MAE_per_target': {'Total Alkalinity': 52.14021301269531, 'Electrical Conductance': 224.76246643066406, 'Dissolved Reactive Phosphorus': 37.244842529296875}, 'RMSE_per_target': {'Total Alkalinity': 66.22492980957031, 'Electrical Conductance': 290.28582763671875, 'Dissolved Reactive Phosphorus': 50.18266677856445}, 'R2_per_target': {'Total Alkalinity': 0.042115747928619385, 'Electrical Conductance': 0.12303739786148071, 'Dissolved Reactive Phosphorus': 0.14201998710632324}}
Aet_Roll6
TEST: {'loss': 32083.194531956375, 'MAE_per_target': {'Total Alkalinity': 54.95713806152344, 'Electrical Conductance': 232.66769409179688, 'Dissolved Reactive Phosphorus': 37.925472259521484}, 'RMSE_per_target': {'Total Alkalinity': 68.07430267333984, 'Electrical Conductance': 298.42718505859375, 'Dissolved Reactive Phosphorus': 50.563758850097656}, 'R2_per_target': {'Total Alkalinity': -0.012130379676818848, 'Electrical Conductance': 0.07315695285797119, 'Dissolved Reactive Phosphorus': 0.12893933057785034}}
Aet_Roll12
TEST: {'loss': 32108.354536335893, 'MAE_per_target': {'Total Alkalinity': 55.8112678527832, 'Electrical Conductance': 232.7908477783203, 'Dissolved Reactive Phosphorus': 38.463890075683594}, 'RMSE_per_target': {'Total Alkalinity': 68.44793701171875, 'Electrical Conductance': 298.3868103027344, 'Dissolved Reactive Phosphorus': 51.041725158691406}, 'R2_per_target': {'Total Alkalinity': -0.02327120304107666, 'Electrical Conductance': 0.07340759038925171, 'Dissolved Reactive Phosphorus': 0.11239367723464966}}
Ppt
TEST: {'loss': 31137.942843721747, 'MAE_per_target': {'Total Alkalinity': 54.86461639404297, 'Electrical Conductance': 228.92755126953125, 'Dissolved Reactive Phosphorus': 37.12601852416992}, 'RMSE_per_target': {'Total Alkalinity': 67.61964416503906, 'Electrical Conductance': 293.7521667480469, 'Dissolved Reactive Phosphorus': 50.50828552246094}, 'R2_per_target': {'Total Alkalinity': 0.001344442367553711, 'Electrical Conductance': 0.10196846723556519, 'Dissolved Reactive Phosphorus': 0.13084954023361206}}
Ppt_Roll3
TEST: {'loss': 30503.989778763564, 'MAE_per_target': {'Total Alkalinity': 53.31842803955078, 'Electrical Conductance': 225.0226593017578, 'Dissolved Reactive Phosphorus': 37.034828186035156}, 'RMSE_per_target': {'Total Alkalinity': 66.58033752441406, 'Electrical Conductance': 290.75811767578125, 'Dissolved Reactive Phosphorus': 50.38585662841797}, 'R2_per_target': {'Total Alkalinity': 0.031806886196136475, 'Electrical Conductance': 0.12018120288848877, 'Dissolved Reactive Phosphorus': 0.1350577473640442}}
Ppt_Roll6
TEST: {'loss': 31979.73455159358, 'MAE_per_target': {'Total Alkalinity': 54.909210205078125, 'Electrical Conductance': 231.79769897460938, 'Dissolved Reactive Phosphorus': 37.93686294555664}, 'RMSE_per_target': {'Total Alkalinity': 68.19593048095703, 'Electrical Conductance': 297.90142822265625, 'Dissolved Reactive Phosphorus': 50.430721282958984}, 'R2_per_target': {'Total Alkalinity': -0.01575016975402832, 'Electrical Conductance': 0.07641971111297607, 'Dissolved Reactive Phosphorus': 0.13351702690124512}}
Ppt_Roll12
TEST: {'loss': 32742.12811864263, 'MAE_per_target': {'Total Alkalinity': 55.21677017211914, 'Electrical Conductance': 235.69496154785156, 'Dissolved Reactive Phosphorus': 38.130741119384766}, 'RMSE_per_target': {'Total Alkalinity': 68.55284118652344, 'Electrical Conductance': 301.5585632324219, 'Dissolved Reactive Phosphorus': 50.88548278808594}, 'R2_per_target': {'Total Alkalinity': -0.02641010284423828, 'Electrical Conductance': 0.053604304790496826, 'Dissolved Reactive Phosphorus': 0.11781930923461914}}
Q
TEST: {'loss': 30937.31640625, 'MAE_per_target': {'Total Alkalinity': 54.40871047973633, 'Electrical Conductance': 227.58624267578125, 'Dissolved Reactive Phosphorus': 38.16733932495117}, 'RMSE_per_target': {'Total Alkalinity': 67.31718444824219, 'Electrical Conductance': 292.8389892578125, 'Dissolved Reactive Phosphorus': 50.25606918334961}, 'R2_per_target': {'Total Alkalinity': 0.010258316993713379, 'Electrical Conductance': 0.10754305124282837, 'Dissolved Reactive Phosphorus': 0.13950824737548828}}
Q_Roll3
TEST: {'loss': 31880.17751186709, 'MAE_per_target': {'Total Alkalinity': 55.5656852722168, 'Electrical Conductance': 231.91246032714844, 'Dissolved Reactive Phosphorus': 37.2818603515625}, 'RMSE_per_target': {'Total Alkalinity': 68.24617004394531, 'Electrical Conductance': 297.2757873535156, 'Dissolved Reactive Phosphorus': 51.08907699584961}, 'R2_per_target': {'Total Alkalinity': -0.017247319221496582, 'Electrical Conductance': 0.08029496669769287, 'Dissolved Reactive Phosphorus': 0.11074596643447876}}
Q_Roll6
TEST: {'loss': 31613.033132487566, 'MAE_per_target': {'Total Alkalinity': 54.98094940185547, 'Electrical Conductance': 230.87596130371094, 'Dissolved Reactive Phosphorus': 38.13029098510742}, 'RMSE_per_target': {'Total Alkalinity': 67.7988052368164, 'Electrical Conductance': 296.1258239746094, 'Dissolved Reactive Phosphorus': 50.51652526855469}, 'R2_per_target': {'Total Alkalinity': -0.003954410552978516, 'Electrical Conductance': 0.08739668130874634, 'Dissolved Reactive Phosphorus': 0.13056594133377075}}
Q_Roll12
TEST: {'loss': 31665.211612087478, 'MAE_per_target': {'Total Alkalinity': 54.70878982543945, 'Electrical Conductance': 229.9929962158203, 'Dissolved Reactive Phosphorus': 37.9727668762207}, 'RMSE_per_target': {'Total Alkalinity': 67.77252197265625, 'Electrical Conductance': 296.3517150878906, 'Dissolved Reactive Phosphorus': 50.775726318359375}, 'R2_per_target': {'Total Alkalinity': -0.003176569938659668, 'Electrical Conductance': 0.08600378036499023, 'Dissolved Reactive Phosphorus': 0.1216210126876831}}
Soil_Roll3
TEST: {'loss': 32186.653802412973, 'MAE_per_target': {'Total Alkalinity': 56.2451286315918, 'Electrical Conductance': 234.2843780517578, 'Dissolved Reactive Phosphorus': 37.8877067565918}, 'RMSE_per_target': {'Total Alkalinity': 69.46484375, 'Electrical Conductance': 298.5384826660156, 'Dissolved Reactive Phosphorus': 51.081932067871094}, 'R2_per_target': {'Total Alkalinity': -0.05390191078186035, 'Electrical Conductance': 0.07246530055999756, 'Dissolved Reactive Phosphorus': 0.11099457740783691}}
Soil_Roll6
TEST: {'loss': 31840.47305181962, 'MAE_per_target': {'Total Alkalinity': 55.64265060424805, 'Electrical Conductance': 233.50100708007812, 'Dissolved Reactive Phosphorus': 37.611873626708984}, 'RMSE_per_target': {'Total Alkalinity': 68.69140625, 'Electrical Conductance': 297.02880859375, 'Dissolved Reactive Phosphorus': 50.76210021972656}, 'R2_per_target': {'Total Alkalinity': -0.030563712120056152, 'Electrical Conductance': 0.08182239532470703, 'Dissolved Reactive Phosphorus': 0.12209224700927734}}
Soil_Roll12
TEST: {'loss': 32643.604451570976, 'MAE_per_target': {'Total Alkalinity': 56.838592529296875, 'Electrical Conductance': 238.74403381347656, 'Dissolved Reactive Phosphorus': 38.642330169677734}, 'RMSE_per_target': {'Total Alkalinity': 70.35173034667969, 'Electrical Conductance': 300.6492004394531, 'Dissolved Reactive Phosphorus': 50.90682601928711}, 'R2_per_target': {'Total Alkalinity': -0.08098506927490234, 'Electrical Conductance': 0.059303343296051025, 'Dissolved Reactive Phosphorus': 0.11707907915115356}}
Tmax
TEST: {'loss': 33604.1885948802, 'MAE_per_target': {'Total Alkalinity': 53.17383575439453, 'Electrical Conductance': 240.05992126464844, 'Dissolved Reactive Phosphorus': 39.4262580871582}, 'RMSE_per_target': {'Total Alkalinity': 68.00687408447266, 'Electrical Conductance': 305.96600341796875, 'Dissolved Reactive Phosphorus': 50.719078063964844}, 'R2_per_target': {'Total Alkalinity': -0.010126233100891113, 'Electrical Conductance': 0.025737762451171875, 'Dissolved Reactive Phosphorus': 0.12357968091964722}}
Tmin
TEST: {'loss': 34171.00392037749, 'MAE_per_target': {'Total Alkalinity': 52.608497619628906, 'Electrical Conductance': 242.78781127929688, 'Dissolved Reactive Phosphorus': 40.66554641723633}, 'RMSE_per_target': {'Total Alkalinity': 67.67346954345703, 'Electrical Conductance': 308.7991943359375, 'Dissolved Reactive Phosphorus': 50.75791931152344}, 'R2_per_target': {'Total Alkalinity': -0.00024628639221191406, 'Electrical Conductance': 0.007611274719238281, 'Dissolved Reactive Phosphorus': 0.12223666906356812}}
Vpd
TEST: {'loss': 32698.20468255538, 'MAE_per_target': {'Total Alkalinity': 55.32778549194336, 'Electrical Conductance': 236.23287963867188, 'Dissolved Reactive Phosphorus': 38.54873275756836}, 'RMSE_per_target': {'Total Alkalinity': 69.19218444824219, 'Electrical Conductance': 301.2203063964844, 'Dissolved Reactive Phosphorus': 50.728641510009766}, 'R2_per_target': {'Total Alkalinity': -0.04564464092254639, 'Electrical Conductance': 0.05572628974914551, 'Dissolved Reactive Phosphorus': 0.1232491135597229}}
Et_Deficit
TEST: {'loss': 31113.991414020118, 'MAE_per_target': {'Total Alkalinity': 54.56345748901367, 'Electrical Conductance': 228.18577575683594, 'Dissolved Reactive Phosphorus': 37.248985290527344}, 'RMSE_per_target': {'Total Alkalinity': 67.37894439697266, 'Electrical Conductance': 293.6158752441406, 'Dissolved Reactive Phosphorus': 50.909427642822266}, 'R2_per_target': {'Total Alkalinity': 0.008441448211669922, 'Electrical Conductance': 0.1028013825416565, 'Dissolved Reactive Phosphorus': 0.11698877811431885}}
Pet_Ppt_Ratio
TEST: {'loss': 30385.099803627938, 'MAE_per_target': {'Total Alkalinity': 53.07488250732422, 'Electrical Conductance': 224.1749267578125, 'Dissolved Reactive Phosphorus': 38.0651741027832}, 'RMSE_per_target': {'Total Alkalinity': 66.32379150390625, 'Electrical Conductance': 290.241455078125, 'Dissolved Reactive Phosphorus': 50.16339874267578}, 'R2_per_target': {'Total Alkalinity': 0.03925371170043945, 'Electrical Conductance': 0.12330549955368042, 'Dissolved Reactive Phosphorus': 0.14267843961715698}}
Runoff_Coefficient
TEST: {'loss': 31214.136143054926, 'MAE_per_target': {'Total Alkalinity': 54.542449951171875, 'Electrical Conductance': 228.68130493164062, 'Dissolved Reactive Phosphorus': 37.637657165527344}, 'RMSE_per_target': {'Total Alkalinity': 67.63408660888672, 'Electrical Conductance': 294.0992431640625, 'Dissolved Reactive Phosphorus': 50.73146438598633}, 'R2_per_target': {'Total Alkalinity': 0.0009175539016723633, 'Electrical Conductance': 0.09984499216079712, 'Dissolved Reactive Phosphorus': 0.12315160036087036}}
Temp_Range
TEST: {'loss': 30968.7009105165, 'MAE_per_target': {'Total Alkalinity': 54.64439392089844, 'Electrical Conductance': 227.68861389160156, 'Dissolved Reactive Phosphorus': 37.835384368896484}, 'RMSE_per_target': {'Total Alkalinity': 68.08086395263672, 'Electrical Conductance': 292.8291931152344, 'Dissolved Reactive Phosphorus': 50.22121047973633}, 'R2_per_target': {'Total Alkalinity': -0.012325406074523926, 'Electrical Conductance': 0.10760277509689331, 'Dissolved Reactive Phosphorus': 0.1407012939453125}}
Aet_Pet_Ration
TEST: {'loss': 30894.26441709991, 'MAE_per_target': {'Total Alkalinity': 54.34390640258789, 'Electrical Conductance': 226.79429626464844, 'Dissolved Reactive Phosphorus': 37.26234817504883}, 'RMSE_per_target': {'Total Alkalinity': 67.08295440673828, 'Electrical Conductance': 292.595947265625, 'Dissolved Reactive Phosphorus': 50.69788360595703}, 'R2_per_target': {'Total Alkalinity': 0.01713395118713379, 'Electrical Conductance': 0.10902374982833862, 'Dissolved Reactive Phosphorus': 0.12431198358535767}}
Temp_Mean
TEST: {'loss': 34775.64447474005, 'MAE_per_target': {'Total Alkalinity': 56.263450622558594, 'Electrical Conductance': 250.72756958007812, 'Dissolved Reactive Phosphorus': 38.36635208129883}, 'RMSE_per_target': {'Total Alkalinity': 68.5477066040039, 'Electrical Conductance': 311.3138732910156, 'Dissolved Reactive Phosphorus': 52.0751953125}, 'R2_per_target': {'Total Alkalinity': -0.026256442070007324, 'Electrical Conductance': -0.008617281913757324, 'Dissolved Reactive Phosphorus': 0.07608598470687866}}
"""

lines = raw_text.strip().splitlines()

results = {}
current_model = None

for line in lines:
    line = line.strip()
    if not line:
        continue

    # Model name line (e.g., "Emis")
    if not line.startswith("TEST:"):
        current_model = line
        results[current_model] = {}
        continue

    # TEST line
    if line.startswith("TEST:"):
        dict_str = line.replace("TEST:", "").strip()
        metrics = ast.literal_eval(dict_str)
        results[current_model] = metrics

# Convert to DataFrame
rows = []

for model_name, metrics in results.items():
    for target in metrics["MAE_per_target"].keys():
        rows.append({
            "Model": model_name,
            "Target": target,
            "MAE": metrics["MAE_per_target"][target],
            "RMSE": metrics["RMSE_per_target"][target],
            "R2": metrics["R2_per_target"][target],
            "Loss": metrics["loss"]
        })

df = pd.DataFrame(rows)
#print(df)

In [ ]:
# # Set display options to show all rows
# pd.set_option('display.max_rows', None)
# # Set display options to show all columns
# pd.set_option('display.max_columns', None)
# print(df)    


In [52]:
display(df)

#df.to_csv('data/feature_comp.csv', index=False)

In [ ]:
# session.sql("""
#     PUT file://output.csv
#     'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/'
#     AUTO_COMPRESS=FALSE
#     OVERWRITE=TRUE
# """).collect()

# print("File saved! Refresh the browser to see the files in the sidebar")